In [4]:
target_cols = 'K_E'
input_folder=f'C:/Users/1/projects/zadachi/3 задача/{target_cols} determ'
output_file=f'{target_cols} determ.csv'

columns = ['K_Nu', 'K_E', 'K_Rho', 'K_incl', 'F', 'K_lambda_r', 'A2', 'А2_surface',  'Nu_host medium', 'E', 'Rho', 'Vp', 'Vs', 'Z']

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')

# Загрузка данных
df = pd.read_csv(output_file)

# Подготовка данных
df['A2/A2_s'] = df['A2'] / df['А2_surface']
y = df[target_cols]
X = df[['A2/A2_s', 'F']]


C:\Users\1\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [6]:
df

,K_Nu,K_E,K_Rho,K_incl_top_position_y,F,K_lambda_r,A2,А2_surface,Nu_host medium,E,Rho,Vp,Vs,Z,A2/A2_s
0,1,0.5000,1,1,0.1,10.000000,5.331410e-13,5.331447e-13,0.25,5.000000e+10,2700,3333.333333,1924.500897,2502.294446,0.999993
1,1,0.5000,1,1,0.2,5.000000,5.334885e-13,5.331474e-13,0.25,5.000000e+10,2700,3333.333333,1924.500897,2502.294446,1.000640
2,1,0.5000,1,1,0.3,3.333333,5.367156e-13,5.363296e-13,0.25,5.000000e+10,2700,3333.333333,1924.500897,2502.294446,1.000720
3,1,0.5000,1,1,0.4,2.500000,5.324516e-13,5.329968e-13,0.25,5.000000e+10,2700,3333.333333,1924.500897,2502.294446,0.998977
4,1,0.5000,1,1,0.5,2.000000,5.332178e-13,5.322721e-13,0.25,5.000000e+10,2700,3333.333333,1924.500897,2502.294446,1.001777
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100095,1,0.5135,1,1,9.6,0.104167,5.526740e-13,5.640118e-13,0.25,5.000000e+10,2700,3378.033616,1950.308618,2502.294446,0.979898
100096,1,0.5135,1,1,9.7,0.103093,5.517025e-13,5.630990e-13,0.25,5.000000e+10,2700,3378.033616,1950.308618,2502.294446,0.979761
100097,1,0.5135,1,1,9.8,0.102041,5.508005e-13,5.622432e-13,0.25,5.000000e+10,2700,3378.033616,1950.308618,2502.294446,0.979648
100098,1,0.5135,1,1,9.9,0.101010,5.499630e-13,5.614406e-13,0.25,5.000000e+10,2700,3378.033616,1950.308618,2502.294446,0.979557


In [7]:
def create_and_train_model(X_train, y_train, X_test, y_test, sample_size):
    """Создает и обучает модель на данных заданного размера"""
    
    scaler_X = StandardScaler()
    
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled = scaler_X.transform(X_test)
    
    # Создание модели
    model = Sequential([
        Dense(128, activation='relu', input_shape=(2,), name='hidden_layer_1'),
        Dropout(0.2),  # Добавляем Dropout для регуляризации
        Dense(64, activation='relu', name='hidden_layer_2'),
        Dropout(0.2),
        Dense(1, name='output_layer')
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    
    # Ранняя остановка
    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=20,
        restore_best_weights=True
    )
    
    # Обучение
    history = model.fit(
        X_train_scaled, y_train,
        validation_split=0.2,
        epochs=100,
        batch_size=min(32, len(X_train)//10),
        callbacks=[early_stopping],
        verbose=0
    )
    
    # Предсказание
    y_pred = np.array(model.predict(X_test_scaled)).flatten()
    y_test = np.array(y_test)
    
    # Метрики
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    # Относительная ошибка
    relative_errors = np.abs((y_test - y_pred) / y_test) * 100
    mean_relative_error = np.mean(relative_errors)
    std_relative_error = np.std(relative_errors)
    
    return {
        'sample_size': sample_size,
        'mse': mse,
        'mae': mae,
        'mape': mape,
        'rmse': rmse,
        'r2': r2,
        'mean_rel_error': mean_relative_error,
        'std_rel_error': std_relative_error,
        'accuracy': 100 - mean_relative_error,
        'history': history.history
    }

# Основной анализ
print("="*70)
print("АНАЛИЗ ВЛИЯНИЯ КОЛИЧЕСТВА ДАННЫХ НА КАЧЕСТВО ПРЕДСКАЗАНИЯ K_E")
print("="*70)

# Размеры выборок для анализа
sample_sizes = [100, 200, 500, 1000, 2000, 5000, 10000, 20000, 50000, 100000]
n_repetitions = 3  # Количество повторений для каждого размера (для усреднения)

results = []

# Фиксируем тестовую выборку (20% от всех данных)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

АНАЛИЗ ВЛИЯНИЯ КОЛИЧЕСТВА ДАННЫХ НА КАЧЕСТВО ПРЕДСКАЗАНИЯ K_E


In [ ]:

print(f"\nТестовая выборка фиксирована: {len(X_test)} образцов")
print(f"Обучающая выборка (полная): {len(X_train_full)} образцов")
print("\n" + "-"*70)

for size in sample_sizes:
    print(f"\nАнализ для размера выборки: {size}")
    
    size_results = []
    
    for rep in range(n_repetitions):
        # Случайно выбираем size образцов из обучающей выборки
        indices = np.random.choice(len(X_train_full), size=min(size, len(X_train_full)), replace=False)
        X_train_sample = X_train_full.iloc[indices]
        y_train_sample = y_train_full.iloc[indices]
        
        # Обучаем и оцениваем модель
        result = create_and_train_model(
            X_train_sample, y_train_sample,
            X_test, y_test,
            size
        )
        size_results.append(result)
        
        print(f"  Повторение {rep+1}/{n_repetitions}: "
              f"Точность = {result['accuracy']:.2f}%")
    
    # Усредняем результаты по повторениям
    avg_result = {
        'sample_size': size,
        'mse': np.mean([r['mse'] for r in size_results]),
        'mae': np.mean([r['mae'] for r in size_results]),
        'mape': np.mean([r['mape'] for r in size_results]),
        'rmse': np.mean([r['rmse'] for r in size_results]),
        'r2': np.mean([r['r2'] for r in size_results]),
        'mean_rel_error': np.mean([r['mean_rel_error'] for r in size_results]),
        'std_rel_error': np.mean([r['std_rel_error'] for r in size_results]),
        'accuracy': np.mean([r['accuracy'] for r in size_results]),
        'accuracy_std': np.std([r['accuracy'] for r in size_results])
    }
    
    results.append(avg_result)
    
    print(f"\n  СРЕДНЕЕ для размера {size}:")
    print(f"    Точность: {avg_result['accuracy']:.2f}% ± {avg_result['accuracy_std']:.2f}%")
    print(f"    R²: {avg_result['r2']:.4f}")
    print(f"    RMSE: {avg_result['rmse']:.6f}")
    print(f"    MAE: {avg_result['mae']:.6f}")
    print(f"    MAPE: {avg_result['mape']:.6f}")

# Создаем DataFrame с результатами
results_df = pd.DataFrame(results)
results_df.to_csv(f'{target_cols} size result.csv', index=False)
print('Датасет сохранен')


Тестовая выборка фиксирована: 20020 образцов
Обучающая выборка (полная): 80080 образцов

----------------------------------------------------------------------

Анализ для размера выборки: 100


626/626 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step
  Повторение 1/3: Точность = 68.97%
626/626 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
  Повторение 2/3: Точность = 75.37%
626/626 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
  Повторение 3/3: Точность = 75.66%

  СРЕДНЕЕ для размера 100:
    Точность: 73.33% ± 3.09%
    R²: 0.3430
    RMSE: 0.349905
    MAE: 0.270001
    MAPE: 0.266672

Анализ для размера выборки: 200
626/626 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
  Повторение 1/3: Точность = 76.61%
626/626 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
  Повторение 2/3: Точность = 73.41%
626/626 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
  Повторение 3/3: Точность = 80.40%

  СРЕДНЕЕ для размера 200:
    Точность: 76.81% ± 2.85%
    R²: 0.4742
    RMSE: 0.313324
    MAE: 0.244380
    MAPE: 0.231945

Анализ для размера выборки: 500
626/626 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step
  Повторение 1/3: Точность = 80.93%
626/626 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
  Повторение 2/3: Точность = 81.52%
626/626 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
  Повторение 3/3: Точ

In [ ]:

# Визуализация результатов
fig, axes = plt.subplots(2, 3, figsize=(18, 12))


# 2. R² от размера выборки
axes[0, 0].plot(results_df['sample_size'], results_df['r2'], 'ro-', linewidth=2, markersize=8)
axes[0, 0].set_xlabel('Размер обучающей выборки')
axes[0, 0].set_ylabel('R² Score')
axes[0, 0].set_title('Зависимость R² от размера выборки')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_xscale('log')

# 3. RMSE от размера выборки
axes[0, 1].plot(results_df['sample_size'], results_df['rmse'], 'go-', linewidth=2, markersize=8)
axes[0, 1].set_xlabel('Размер обучающей выборки')
axes[0, 1].set_ylabel('RMSE')
axes[0, 1].set_title('Зависимость RMSE от размера выборки')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_xscale('log')

# 3.2 MAE от размера выборки
axes[0, 2].plot(results_df['sample_size'], results_df['mae'], 'go-', linewidth=2, markersize=8)
axes[0, 2].set_xlabel('Размер обучающей выборки')
axes[0, 2].set_ylabel('MAE')
axes[0, 2].set_title('Зависимость MAE от размера выборки')
axes[0, 2].grid(True, alpha=0.3)
axes[0, 2].set_xscale('log')

# 4. Относительная ошибка от размера выборки
axes[1, 0].plot(results_df['sample_size'], results_df['mean_rel_error'], 'mo-', linewidth=2, markersize=8)
axes[1, 0].fill_between(results_df['sample_size'],
                        results_df['mean_rel_error'] - results_df['std_rel_error'],
                        results_df['mean_rel_error'] + results_df['std_rel_error'],
                        alpha=0.3, color='m')
axes[1, 0].set_xlabel('Размер обучающей выборки')
axes[1, 0].set_ylabel(' MAPE (%)')
axes[1, 0].set_title('Зависимость MAPE от размера выборки')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_xscale('log')

# 5. Скорость обучения (время/качество)
improvement = []
for i in range(1, len(results_df)):
    imp = (results_df['accuracy'].iloc[i] - results_df['accuracy'].iloc[i-1]) / \
          (results_df['sample_size'].iloc[i] - results_df['sample_size'].iloc[i-1])
    improvement.append(imp * 1000)  # Умножаем для лучшей визуализации

axes[1, 1].plot(results_df['sample_size'].iloc[1:], improvement, 'co-', linewidth=2, markersize=8)
axes[1, 1].set_xlabel('Размер обучающей выборки')
axes[1, 1].set_ylabel('Улучшение точности на 1000 образцов (%)')
axes[1, 1].set_title('Скорость улучшения качества')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_xscale('log')
axes[1, 1].axhline(y=0, color='r', linestyle='--', alpha=0.5)

# 6. Сравнение метрик (нормализованные)
metrics_normalized = pd.DataFrame({
    'Точность': results_df['accuracy'] / results_df['accuracy'].max(),
    'R²': results_df['r2'] / results_df['r2'].max(),
    'RMSE (инвертир.)': 1 - results_df['rmse'] / results_df['rmse'].max()
})

for metric in metrics_normalized.columns:
    axes[1, 2].plot(results_df['sample_size'], metrics_normalized[metric], 
                   'o-', linewidth=2, markersize=6, label=metric)

axes[1, 2].set_xlabel('Размер обучающей выборки')
axes[1, 2].set_ylabel('Нормализованные метрики')
axes[1, 2].set_title('Сравнение метрик (нормализованные)')
axes[1, 2].grid(True, alpha=0.3)
axes[1, 2].set_xscale('log')
axes[1, 2].legend()
axes[1, 2].set_ylim([0, 1.1])

plt.tight_layout()
plt.show()


NameError: name 'plt' is not defined

In [ ]:
results_df

,sample_size,mse,mae,mape,rmse,r2,mean_rel_error,std_rel_error,accuracy,accuracy_std
0,100,0.126281,0.284629,0.280754,0.354167,0.326115,28.075424,30.196695,71.924576,3.633431
1,200,0.107724,0.258603,0.248458,0.327014,0.425141,24.845792,26.351228,75.154208,1.946982
2,500,0.070740,0.195851,0.189830,0.265746,0.622503,18.983012,23.623760,81.016988,1.819188
3,1000,0.055528,0.160172,0.155014,0.235590,0.703682,15.501380,21.934474,84.498620,0.886945
4,2000,0.044849,0.130607,0.124981,0.211747,0.760665,12.498064,20.297348,87.501936,0.495597
5,5000,0.035810,0.108580,0.103615,0.189218,0.808902,10.361513,19.033145,89.638487,0.395603
6,10000,0.031714,0.096540,0.093341,0.178074,0.830763,9.334146,18.403348,90.665854,0.233010
7,20000,0.028841,0.091565,0.089258,0.169824,0.846095,8.925783,17.879963,91.074217,0.234535
8,50000,0.027678,0.087518,0.084455,0.166356,0.852300,8.445540,17.464175,91.554460,0.229636
9,100000,0.026149,0.086846,0.084557,0.161697,0.860458,8.455745,17.291958,91.544255,0.252481
